In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

engine = 

print("Подключение создано")

Подключение создано


In [61]:
import socket

s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.settimeout(5)
try:
    s.connect(("thomas.proxy.rlwy.net", 51432))
    print("Порт открыт, соединение установлено")
except Exception as e:
    print("Не удалось подключиться:", e)
finally:
    s.close()

Порт открыт, соединение установлено


In [62]:
query = text("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = :schema AND table_name LIKE :pattern
    ORDER BY table_name
""")

with engine.connect() as conn:
    tables = pd.read_sql(query, conn, params={"schema": "public", "pattern": "scoring_%"} )

tables

,table_name
0,scoring_applications
1,scoring_applications_full
2,scoring_bureau_inquiries
3,scoring_bureau_loans
4,scoring_bureau_snapshots
5,scoring_card_disputes
6,scoring_card_transactions
7,scoring_cards
8,scoring_clients_old
9,scoring_collection_actions


In [63]:
table_names = tables["table_name"].tolist()

with engine.connect() as conn:
    for table_name in table_names:
        # Экранируем имя двойными кавычками и экранируем возможные кавычки внутри имени
        safe_table_name = table_name.replace('"', '""')
        query = text(f'SELECT COUNT(*) AS row_count FROM public."{safe_table_name}"')
        count = pd.read_sql(query, conn).iloc[0, 0]
        print(table_name, count)

scoring_applications 18000
scoring_applications_full 11277
scoring_bureau_inquiries 31383
scoring_bureau_loans 34684
scoring_bureau_snapshots 86161
scoring_card_disputes 2467
scoring_card_transactions 1044743
scoring_cards 12614
scoring_clients_old 6000
scoring_collection_actions 9135
scoring_contracts 11277
scoring_customers 8000
scoring_model_v1_scores 18000
scoring_payments 103150
scoring_payments_backup 73421
scoring_temp 0
scoring_txn_sample 5000


In [67]:
query = text("""
    SELECT
        table_name,
        ordinal_position,
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_schema = :schema
      AND table_name LIKE :pattern
    ORDER BY table_name, ordinal_position
""")

with engine.connect() as conn:
    columns = pd.read_sql(query, conn, params={"schema": "public", "pattern": "scoring_%"})

columns

,table_name,ordinal_position,column_name,data_type,is_nullable
0,scoring_applications,1,application_id,bigint,NO
1,scoring_applications,2,client_id,bigint,NO
2,scoring_applications,3,decision_date,date,NO
3,scoring_applications,4,channel,text,YES
4,scoring_applications,5,purpose,text,YES
...,...,...,...,...,...
118,scoring_txn_sample,7,txn_status,text,YES
119,scoring_txn_sample,8,decline_reason,text,YES
120,scoring_txn_sample,9,is_recurring,boolean,YES
121,scoring_txn_sample,10,entry_mode,text,YES


| Таблица                   | Роль               | Решение                  | Обоснование                                                      |
| ------------------------- | ------------------ | ------------------------ | ---------------------------------------------------------------- |
| scoring_applications      | Заявки             | Использовать             | Есть application_id, client_id, decision_date, анкета и решение  |
| scoring_contracts         | Выданные договоры  | Использовать             | Связывает заявку с договором и содержит issue_date               |
| scoring_payments          | График и оплаты    | Использовать             | Нужна для вычисления просрочки                                   |
| scoring_bureau_snapshots  | Снимки бюро        | Использовать для фич     | Можно выбрать последний снимок до decision_date                  |
| scoring_bureau_loans      | Кредиты бюро       | Использовать осторожно   | Только исторические данные, доступные на дату решения            |
| scoring_bureau_inquiries  | Запросы в бюро     | Использовать для фич     | Можно считать запросы до заявки                                  |
| scoring_cards             | Карты клиентов     | Использовать для связи   | Связывает клиента и карты                                        |
| scoring_card_transactions | Транзакции         | Использовать для фич     | Только транзакции до decision_date                               |
| scoring_applications_full | Обогащённая заявка | Не использовать напрямую | Содержит признаки будущего: dpd_max, collection и текущий статус |
| scoring_payments_backup   | Backup             | Исключить                | Явно обозначена как backup                                       |
| scoring_clients_old       | Старый справочник  | Исключить                | Старая версия данных                                             |
| scoring_temp              | Служебная          | Исключить                | Нет бизнес-связи с задачей                                       |
| scoring_model_v1_scores   | Старая модель      | Не использовать как фичу | Может содержать решение старой модели и историческую логику      |